# 01 — Agent Profiles

Inspect persona text, narrative text, and demographic attributes generated from YouGov survey data.

**Covers:** Task A (narrative fix), Issue 1 (climate policy opinion system)

In [1]:
import sys, os, random
sys.path.insert(0, os.path.abspath("../src"))

from gabm.abm.attributes.gender import GenderID, GenderMap
from gabm.abm.attributes.ethnicity import EthnicityID
from gabm.abm.attributes.income import IncomeID
from gabm.abm.attributes.politics import PoliticsID
from gabm.abm.attributes.education import EducationID
from gabm.abm.attributes.region import RegionID
from gabm.abm.attributes.family import FamilyID
from gabm.abm.democracy.election import ElectionID

from cag.io.survey import load
from cag.abm.environment import SurveyedNation
from cag.abm.agent import SurveyedCitizen
from cag.abm.attributes.ethnicity import SurveyEthnicityMap
from cag.abm.attributes.income import SurveyIncomeMap
from cag.abm.attributes.politics import SurveyPoliticsMap
from cag.abm.attributes.education import SurveyEducationMap
from cag.abm.attributes.region import UKRegionMap
from cag.abm.attributes.family import SurveyFamilyMap
from cag.abm.democracy.elections.ukge2019 import UKGE2019VoteID, UKGE2019, UKGE2019VoteMap
from cag.abm.democracy.elections.brexit import BrexitVoteID, Brexit, BrexitVoteMap
from cag.abm.attributes.narratives import (
    NarrativeAttributeID, SelftranscMap, SelfenhMap,
    OpennessMap, ConformTradMap, SDOMap, EDOMap, RWAMap,
    rescale_1_6, rescale_1_7
)

print("Imports OK")

Imports OK


## Setup: Create environment and load agents from survey data

In [2]:
random.seed(42)
year = 2026

# Elections
UKGE2019_ELECTION_ID = ElectionID(0)
BREXIT_REFERENDUM_ID = ElectionID(1)

# Attribute maps
surveyed_nation = SurveyedNation(
    year=year, place="UK",
    gender_map=GenderMap(),
    region_map=UKRegionMap(),
    education_map=SurveyEducationMap(),
    ethnicity_map=SurveyEthnicityMap(),
    income_map=SurveyIncomeMap(),
    politics_map=SurveyPoliticsMap(),
    family_map=SurveyFamilyMap(),
    ukge2019_vote_map=UKGE2019VoteMap(UKGE2019_ELECTION_ID),
    brexit_vote_map=BrexitVoteMap(BREXIT_REFERENDUM_ID),
    selftransc_map=SelftranscMap,
    selfenh_map=SelfenhMap,
    openness_map=OpennessMap,
    conformtrad_map=ConformTradMap,
    sdo_map=SDOMap,
    edo_map=EDOMap,
    rwa_map=RWAMap,
)

# Load survey data
data = load("../data/yougov_survey_data/YouGovProcessedData.csv")
print(f"Loaded {len(data)} survey rows")

Loaded 1086 survey rows


In [3]:
# Create agents from survey rows
agents = []
for i in range(len(data)):
    row = data.iloc[i]
    agent_id = row.get('ID', None)
    age = int(row.get('age', 0))
    year_of_birth = year - age
    gender_id = GenderID.MALE if int(row.get('male_dummy', 0)) == 1 else GenderID.FEMALE
    region_id = RegionID(int(row.get('tprofile_GOR', 0)))
    education_id = EducationID(int(row.get('profile_education_level', 0)))
    income_id = IncomeID(int(row.get('tprofile_gross_household', 0)))
    ethnicity_id = EthnicityID(int(row.get('ethnicity_R', 0)))
    family_id = FamilyID.PARENT if int(row.get('parent_dummy', 0)) == 1 else FamilyID.NOT_PARENT
    ukge2019_vote_id = UKGE2019VoteID(int(row.get('Vote2019R', 0)))
    brexit_vote_id = BrexitVoteID(int(row.get('pastvote_EURef', 0)))
    politics_id = PoliticsID(int(row.get('Political_Left_Right', 0)))
    selftransc_id = rescale_1_6(int(row.get('Selftransc_Val', 0)))
    selfenh_id = rescale_1_6(int(row.get('Selfenh_Values', 0)))
    openness_id = rescale_1_6(int(row.get('Openness', 0)))
    conformtrad_id = rescale_1_6(int(row.get('ConformTrad', 0)))
    sdo_id = rescale_1_7(int(row.get('SDO', 0)))
    edo_id = rescale_1_7(int(row.get('EDO', 0)))
    rwa_id = rescale_1_6(int(row.get('RWA', 0)))

    sc = SurveyedCitizen(
        agent_id=agent_id, environment=surveyed_nation,
        year_of_birth=year_of_birth, gender_id=gender_id,
        opinions=None, region_id=region_id, ethnicity_id=ethnicity_id,
        income_id=income_id, education_id=education_id,
        politics_id=politics_id, family_id=family_id,
        ukge2019_vote_id=ukge2019_vote_id, brexit_vote_id=brexit_vote_id,
        selftransc_id=selftransc_id, selfenh_id=selfenh_id,
        openness_id=openness_id, conformtrad_id=conformtrad_id,
        sdo_id=sdo_id, edo_id=edo_id, rwa_id=rwa_id,
    )
    agents.append(sc)

for sc in agents:
    surveyed_nation.agents_active[sc.id] = sc

print(f"Created {len(agents)} agents")

Created 1086 agents


## Sample Persona + Narrative Output

Print the full persona and narrative text for 5 randomly selected agents. This is the text that will be used as the LLM system prompt in later issues.

In [4]:
sample = random.sample(agents, 5)

for i, agent in enumerate(sample, 1):
    persona = agent.get_persona()
    narrative = agent.get_narrative()
    print(f"{'='*80}")
    print(f"Agent {i} (ID: {agent.id})")
    print(f"{'='*80}")
    print(f"\n[PERSONA]\n{persona}")
    print(f"\n[NARRATIVE]\n{narrative}")
    print(f"\n[COMBINED — LLM system prompt]\n{persona}\n\n{narrative}")
    print()

Agent 1 (ID: 430.0)

[PERSONA]
I am a 68 year old female living in the East of England. My ethnicity is white. I have a no formal qualifications. My gross household income is £10,000 - £14,999 per year. I am a parent. I position myself centre of the political spectrum. I voted for the Brexit party candidate in the 2019 General Election. I don't know what I voted in the 2016 EU Referendum.

[NARRATIVE]
When it comes to my core values and worldview: I care about the people close to me and have a basic respect for nature, but I do not actively champion global equality or make environmental protection a primary, driving life focus. I am not strongly driven by the need to get ahead of others, impress people, or hold leadership positions where I tell others what to do. I prefer routine and the familiar, showing little interest in taking risks, seeking out new adventures, or coming up with highly original ideas. I maintain a general respect for elders and standard societal norms, but I am fle

## Climate Policy Opinion System (Issue 1)

The 6 climate policies, A–G response scale, and opinion shift clamping.

In [5]:
from cag.abm.attributes.opinion import (
    ClimatePolicyID, ALL_CLIMATE_POLICIES,
    SURVEY_QUESTIONS, RESPONSE_SCALE, RESPONSE_LABELS,
    SURVEY_COLUMN_MAP, clamp_opinion_shift,
)

# Print all 6 policies and their survey questions
for policy in ALL_CLIMATE_POLICIES:
    col = SURVEY_COLUMN_MAP[policy]
    q = SURVEY_QUESTIONS[policy]
    print(f"[{policy.id}] {col}")
    print(f"  {q}\n")

# Print response scale
print("Response Scale:")
for letter in "ABCDEFG":
    print(f"  {letter} = {RESPONSE_LABELS[letter]} ({RESPONSE_SCALE[letter]:+d})")

[1] page5posttreatment6_1
  Please say how much you support or oppose government policies that do the following: Accelerate the roll-out of renewable energy production, (e.g. more offshore and onshore wind parks)

[2] page5posttreatment6_4
  Please say how much you support or oppose government policies that do the following: Ban new oil/gas/coal licenses

[3] page5posttreatment6_5
  Please say how much you support or oppose government policies that do the following: Ban the sale of new petrol cars by no later than 2030

[4] page5posttreatment6_7
  Please say how much you support or oppose government policies that do the following: Mandate that all new housing developments should have non-fossil fuel heating systems, roof-top solar panels, high-level of insulation

[5] page5posttreatment6_9
  Please say how much you support or oppose government policies that do the following: Impose a carbon tax on fossil fuel sale and distribute the tax revenues to the public (i.e. carbon fee and divid

In [6]:
# Demonstrate opinion shift clamping
examples = [
    (-1, 2, 1, "shift +3, clamped to +1 → 0"),
    (1, -2, 1, "shift -3, clamped to -1 → 0"),
    (2, 2, 1, "no change → 2"),
    (-3, 3, 1, "shift +6, clamped to +1 → -2"),
    (0, 1, 1, "shift +1, within limit → 1"),
]

print("Clamping examples (previous, raw_new, max_shift → clamped):")
for prev, new, ms, desc in examples:
    result = clamp_opinion_shift(prev, new, max_shift=ms)
    print(f"  clamp({prev:+d}, {new:+d}, max_shift={ms}) = {result:+d}  ({desc})")

Clamping examples (previous, raw_new, max_shift → clamped):
  clamp(-1, +2, max_shift=1) = +0  (shift +3, clamped to +1 → 0)
  clamp(+1, -2, max_shift=1) = +0  (shift -3, clamped to -1 → 0)
  clamp(+2, +2, max_shift=1) = +2  (no change → 2)
  clamp(-3, +3, max_shift=1) = -2  (shift +6, clamped to +1 → -2)
  clamp(+0, +1, max_shift=1) = +1  (shift +1, within limit → 1)
